# [16.7] Data Shapley in One Training Run - Solutions

## Core question

Can exact Data Shapley, sampled permutation Data Shapley, and a one-run gradient-dot proxy all identify the same harmful training example?

## Learning objectives

By the end, you should be able to:

1. Define training-example utility as validation-loss improvement after one step.
2. Enumerate complete training-example coalition tables.
3. Compute exact Data Shapley values for a tiny training run.
4. Approximate Data Shapley with sampled training-example permutations.
5. Compare exact values with one-run gradient-dot scores.
6. Reject random-data and label-shuffled controls.
7. Interpret the CUDA report without claiming production-scale data valuation.

> Difficulty: 4/5  
> Importance: 4/5

<img src="../../instructions/assets/data_shapley_validation_loop.svg" width="760">

The toy problem has three helpful examples and one flipped-label harmful example. Exact values should be `[0.6412, 0.6412, 0.6412, -1.1736]`.

<details><summary>Help - why start this small?</summary>

All 16 coalitions can be enumerated, so the exact harmful-example target is not ambiguous. Larger data-attribution methods should earn trust against this toy oracle before making real-data claims.

</details>


## Setup

Run this once. The tests are deterministic and small; the final CUDA cell can rerun the one-step model-organism path from the solution module.

<details><summary>Expected output</summary>

No printed output. Imports should succeed.

</details>


In [1]:
from collections.abc import Callable, Mapping
import itertools
import json
import random
import sys
import time
from pathlib import Path

import torch as t

chapter = "chapter16_shapley_attribution_baselines"
section = "part7_data_shapley_in_one_training_run"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part7_data_shapley_in_one_training_run.tests as tests

Coalition = frozenset[int]
MAIN = True


## Toy problem

The fourth label is flipped. This is the known harmful example.

<details><summary>Expected output</summary>

No printed output. `toy_data_shapley_problem` should define four training examples and one validation example.

</details>


In [2]:
DATA_SHAPLEY_LR = 0.5
DATA_SHAPLEY_MC_SAMPLES = 512
DATA_SHAPLEY_RANDOM_CONTROL_SEED = 13
DATA_SHAPLEY_LABEL_SHUFFLE_PERMUTATION = (0, 3, 2, 1)
DATA_SHAPLEY_RUNTIME_REPEATS = 128


def toy_data_shapley_problem() -> tuple[t.Tensor, t.Tensor, t.Tensor, t.Tensor]:
    train_x = t.ones(4, 1, dtype=t.float64)
    train_y = t.tensor([1.0, 1.0, 1.0, -1.0], dtype=t.float64)
    val_x = t.ones(1, 1, dtype=t.float64)
    val_y = t.ones(1, dtype=t.float64)
    return train_x, train_y, val_x, val_y


## Exercise 1 - one-step utility

Implement validation-loss improvement after one gradient step on a coalition.

<details><summary>Expected output</summary>

```text
All tests in `test_one_step_linear_utility_toy_oracle` passed!
```

</details>

<details><summary>What you should see</summary>

```text
v(empty) = 0.0
v({helpful}) = 1.0
v({flipped}) = -3.0
v(all) = 0.75
```

</details>

<details><summary>Common bugs</summary>

- Using training loss instead of validation-loss improvement.
- Updating on the validation example.
- Letting the empty coalition train.

</details>

<details><summary>Solution</summary>

Compute the baseline validation loss at zero weight, take one analytic gradient step on the selected training examples, and subtract the updated validation loss.

</details>


In [3]:
def one_step_linear_utility(
    train_x: t.Tensor,
    train_y: t.Tensor,
    val_x: t.Tensor,
    val_y: t.Tensor,
    coalition: Coalition,
    *,
    learning_rate: float = DATA_SHAPLEY_LR,
) -> float:
    """Return validation-loss improvement after one step on `coalition`."""
    train_x = train_x.double()
    train_y = train_y.double()
    val_x = val_x.double()
    val_y = val_y.double()
    weight = t.zeros(train_x.shape[1], dtype=t.float64)
    baseline_loss = ((val_x @ weight - val_y) ** 2).mean()
    if not coalition:
        return 0.0
    indices = t.tensor(sorted(coalition), dtype=t.long)
    selected_x = train_x[indices]
    selected_y = train_y[indices]
    train_error = selected_x @ weight - selected_y
    gradient = (2 * train_error.unsqueeze(-1) * selected_x).mean(dim=0)
    updated_weight = weight - learning_rate * gradient
    updated_loss = ((val_x @ updated_weight - val_y) ** 2).mean()
    return float((baseline_loss - updated_loss).item())


if MAIN:
    tests.test_one_step_linear_utility_toy_oracle(one_step_linear_utility)


All tests in `test_one_step_linear_utility_toy_oracle` passed!


## Exercise 2 - exact Data Shapley

Enumerate every training-example coalition and compute exact weighted marginal contributions.

<details><summary>Expected output</summary>

```text
All tests in `test_data_coalition_values_complete_table` passed!
All tests in `test_exact_data_shapley_values_matches_hand_checked_result` passed!
```

</details>

<details><summary>What you should see</summary>

```text
exact_values = [0.6412, 0.6412, 0.6412, -1.1736]
harmful_index = 3
```

</details>

<details><summary>Common bugs</summary>

- Treating features as players instead of training examples.
- Missing the empty or full coalition.
- Forgetting Shapley's factorial weights.

</details>

<details><summary>Solution</summary>

Evaluate all `2**4` coalitions with your utility function, then apply exact Shapley's weighted marginal formula.

</details>


In [4]:
def all_coalitions(num_players: int) -> tuple[Coalition, ...]:
    """Return every training-example coalition."""
    if num_players <= 0:
        raise ValueError("num_players must be positive.")
    coalitions: list[Coalition] = []
    for size in range(num_players + 1):
        coalitions.extend(
            frozenset(group) for group in itertools.combinations(range(num_players), size)
        )
    return tuple(coalitions)


def data_coalition_values(
    train_x: t.Tensor,
    train_y: t.Tensor,
    val_x: t.Tensor,
    val_y: t.Tensor,
    *,
    learning_rate: float = DATA_SHAPLEY_LR,
) -> dict[Coalition, float]:
    """Evaluate one-step utility on every training-example coalition."""
    return {
        coalition: one_step_linear_utility(
            train_x,
            train_y,
            val_x,
            val_y,
            coalition,
            learning_rate=learning_rate,
        )
        for coalition in all_coalitions(int(train_x.shape[0]))
    }


def exact_shapley_values(
    coalition_values: Mapping[Coalition | tuple[int, ...], float],
    *,
    num_players: int,
) -> t.Tensor:
    """Compute exact Shapley values from a complete coalition table."""
    values = {frozenset(key): float(value) for key, value in coalition_values.items()}
    expected = set(all_coalitions(num_players))
    missing = expected - set(values)
    if missing:
        raise ValueError(f"coalition table is missing {len(missing)} coalitions.")
    shapley = t.zeros(num_players, dtype=t.float64)
    factorial = __import__("math").factorial
    denominator = factorial(num_players)
    for player in range(num_players):
        others = [candidate for candidate in range(num_players) if candidate != player]
        for size in range(num_players):
            weight = factorial(size) * factorial(num_players - size - 1) / denominator
            for group in itertools.combinations(others, size):
                coalition = frozenset(group)
                shapley[player] += weight * (values[coalition | {player}] - values[coalition])
    return shapley


def exact_data_shapley_values(
    train_x: t.Tensor,
    train_y: t.Tensor,
    val_x: t.Tensor,
    val_y: t.Tensor,
    *,
    learning_rate: float = DATA_SHAPLEY_LR,
) -> t.Tensor:
    """Compute exact Data Shapley values for the one-step problem."""
    values = data_coalition_values(
        train_x,
        train_y,
        val_x,
        val_y,
        learning_rate=learning_rate,
    )
    return exact_shapley_values(values, num_players=int(train_x.shape[0]))


if MAIN:
    tests.test_data_coalition_values_complete_table(data_coalition_values)
    tests.test_exact_data_shapley_values_matches_hand_checked_result(
        exact_data_shapley_values
    )


All tests in `test_data_coalition_values_complete_table` passed!
All tests in `test_exact_data_shapley_values_matches_hand_checked_result` passed!


## Exercise 3 - sampled permutation Data Shapley

Estimate Data Shapley by averaging marginal utility over sampled training-example orderings.

<details><summary>Expected output</summary>

```text
All tests in `test_sampled_permutation_data_shapley_approximates_exact` passed!
```

</details>

<details><summary>Common bugs</summary>

- Sampling arbitrary coalitions instead of permutations.
- Forgetting to update the running coalition.
- Using too few samples and losing the harmful-example ranking.

</details>

<details><summary>Solution</summary>

For each sampled ordering, walk from empty coalition to full coalition and add each marginal utility jump to the entering example.

</details>


In [5]:
def sampled_permutation_shapley_values(
    coalition_values: Mapping[Coalition | tuple[int, ...], float],
    *,
    num_players: int,
    num_samples: int,
    seed: int = 0,
) -> t.Tensor:
    """Estimate Shapley values by sampling random player orderings."""
    if num_samples <= 0:
        raise ValueError("num_samples must be positive.")
    values = {frozenset(key): float(value) for key, value in coalition_values.items()}
    rng = random.Random(seed)
    totals = t.zeros(num_players, dtype=t.float64)
    players = tuple(range(num_players))
    for _ in range(num_samples):
        coalition: Coalition = frozenset()
        for player in rng.sample(players, k=num_players):
            with_player = coalition | {player}
            totals[player] += values[with_player] - values[coalition]
            coalition = with_player
    return totals / num_samples


if MAIN:
    tests.test_sampled_permutation_data_shapley_approximates_exact(
        sampled_permutation_shapley_values
    )


All tests in `test_sampled_permutation_data_shapley_approximates_exact` passed!


## Exercise 4 - in-run first-order scores

Compute per-example train gradients and the validation gradient at initialization, then take gradient-dot scores.

<details><summary>Expected output</summary>

```text
All tests in `test_in_run_first_order_scores_toy_oracle` passed!
```

</details>

<details><summary>What you should see</summary>

```text
gradient_scores = [4.0, 4.0, 4.0, -4.0]
```

</details>

<details><summary>Solution</summary>

At zero weight, compute the validation gradient and the per-example training gradients, then return the dot products.

</details>


In [6]:
def in_run_first_order_data_scores(
    train_x: t.Tensor,
    train_y: t.Tensor,
    val_x: t.Tensor,
    val_y: t.Tensor,
) -> t.Tensor:
    """Compute per-example gradient-dot scores from initialization."""
    train_x = train_x.double()
    train_y = train_y.double()
    val_x = val_x.double()
    val_y = val_y.double()
    weight = t.zeros(train_x.shape[1], dtype=t.float64)
    val_error = val_x @ weight - val_y
    val_gradient = (2 * val_error.unsqueeze(-1) * val_x).mean(dim=0)
    train_error = train_x @ weight - train_y
    train_gradients = 2 * train_error.unsqueeze(-1) * train_x
    return train_gradients @ val_gradient


if MAIN:
    tests.test_in_run_first_order_scores_toy_oracle(in_run_first_order_data_scores)


All tests in `test_in_run_first_order_scores_toy_oracle` passed!


## Exercise 5 - reports and controls

Package exact, sampled, in-run, random-data, and label-shuffled reports.

<details><summary>Expected output</summary>

```text
All tests in `test_exact_data_shapley_smoke_test` passed!
All tests in `test_monte_carlo_data_shapley_smoke_test` passed!
All tests in `test_in_run_data_shapley_smoke_test` passed!
All tests in `test_random_data_attribution_failure_smoke_test` passed!
All tests in `test_label_shuffled_attribution_failure_smoke_test` passed!
```

</details>

<details><summary>Help - why must controls fail?</summary>

If random data or shuffled labels preserve the planted signal, the method is measuring the wrong thing. A failed control is required before the positive toy result is meaningful.

</details>

<details><summary>Solution</summary>

Use exact values as the signal, then verify random-data and label-shuffled variants do not recover the planted harmful index or correlation.

</details>


In [7]:
def pearson_correlation(first: t.Tensor, second: t.Tensor) -> float:
    first = first.double().flatten()
    second = second.double().flatten()
    first_centered = first - first.mean()
    second_centered = second - second.mean()
    denominator = first_centered.norm() * second_centered.norm()
    if float(denominator.item()) == 0.0:
        return 0.0
    return float((first_centered @ second_centered / denominator).item())


def _json_list(value):
    return value.tolist() if hasattr(value, "tolist") else value


def exact_data_shapley_smoke_test() -> dict:
    """Return exact Data Shapley metrics for the toy problem."""
    train_x, train_y, val_x, val_y = toy_data_shapley_problem()
    exact = exact_data_shapley_values(train_x, train_y, val_x, val_y)
    full = frozenset(range(int(train_x.shape[0])))
    values = data_coalition_values(train_x, train_y, val_x, val_y)
    harmful_index = int(exact.argmin().item())
    helpful_index = int(exact.argmax().item())
    harmful_removal_delta = values[full - {harmful_index}] - values[full]
    helpful_addition_utility = values[frozenset({helpful_index})]
    return {
        "exact_values": exact.tolist(),
        "full_utility": values[full],
        "baseline_utility": values[frozenset()],
        "harmful_index": harmful_index,
        "harmful_value": float(exact[harmful_index].item()),
        "helpful_index": helpful_index,
        "helpful_value": float(exact[helpful_index].item()),
        "harmful_removal_delta": harmful_removal_delta,
        "helpful_addition_utility": helpful_addition_utility,
        "deletion_test_passes": harmful_removal_delta > 0.0,
        "addition_test_passes": helpful_addition_utility > 0.0,
    }


def monte_carlo_data_shapley_smoke_test() -> dict:
    """Return sampled permutation Data Shapley metrics."""
    train_x, train_y, val_x, val_y = toy_data_shapley_problem()
    values = data_coalition_values(train_x, train_y, val_x, val_y)
    exact = exact_data_shapley_values(train_x, train_y, val_x, val_y)
    sampled = sampled_permutation_shapley_values(
        values,
        num_players=4,
        num_samples=DATA_SHAPLEY_MC_SAMPLES,
        seed=0,
    )
    max_abs_error = float((sampled - exact).abs().max().item())
    exact_top_value = float(exact.max().item())
    sampled_top = int(sampled.argmax().item())
    return {
        "exact_values": exact.tolist(),
        "sampled_values": sampled.tolist(),
        "max_abs_error": max_abs_error,
        "top_example_matches": float(exact[sampled_top].item()) == exact_top_value,
        "harmful_example_matches": int(sampled.argmin().item()) == int(exact.argmin().item()),
        "approximates_exact": max_abs_error < 0.08,
    }


def in_run_data_shapley_smoke_test() -> dict:
    """Return exact-vs-in-run proxy metrics."""
    train_x, train_y, val_x, val_y = toy_data_shapley_problem()
    exact = exact_data_shapley_values(train_x, train_y, val_x, val_y)
    scores = in_run_first_order_data_scores(train_x, train_y, val_x, val_y)
    correlation = pearson_correlation(exact, scores)
    harmful_index = int(exact.argmin().item())
    helpful_index = int(exact.argmax().item())
    return {
        "exact_values": exact.tolist(),
        "in_run_scores": scores.tolist(),
        "pearson_correlation": correlation,
        "harmful_index": harmful_index,
        "in_run_harmful_index": int(scores.argmin().item()),
        "helpful_index": helpful_index,
        "in_run_helpful_index": int(scores.argmax().item()),
        "identifies_harmful": int(scores.argmin().item()) == harmful_index,
        "identifies_helpful": int(scores.argmax().item()) == helpful_index,
        "correlates_with_exact": correlation > 0.99,
    }


def _random_data_shapley_problem() -> tuple[t.Tensor, t.Tensor, t.Tensor, t.Tensor]:
    generator = t.Generator().manual_seed(DATA_SHAPLEY_RANDOM_CONTROL_SEED)
    train_x = t.randn(4, 3, generator=generator, dtype=t.float64)
    train_y = t.randn(4, generator=generator, dtype=t.float64)
    val_x = t.randn(2, 3, generator=generator, dtype=t.float64)
    val_y = t.randn(2, generator=generator, dtype=t.float64)
    return train_x, train_y, val_x, val_y


def _label_shuffled_data_shapley_problem() -> tuple[t.Tensor, t.Tensor, t.Tensor, t.Tensor]:
    train_x, train_y, val_x, val_y = toy_data_shapley_problem()
    permutation = t.tensor(DATA_SHAPLEY_LABEL_SHUFFLE_PERMUTATION, dtype=t.long)
    return train_x, train_y[permutation], val_x, val_y


def _signal_failure_metrics(
    train_x: t.Tensor,
    train_y: t.Tensor,
    val_x: t.Tensor,
    val_y: t.Tensor,
    *,
    signal_exact: t.Tensor,
) -> dict:
    exact = exact_data_shapley_values(train_x, train_y, val_x, val_y)
    scores = in_run_first_order_data_scores(train_x, train_y, val_x, val_y)
    exact_signal_correlation = pearson_correlation(signal_exact, exact)
    in_run_signal_correlation = pearson_correlation(signal_exact, scores)
    return {
        "exact_values": exact.tolist(),
        "in_run_scores": scores.tolist(),
        "harmful_index": int(exact.argmin().item()),
        "helpful_index": int(exact.argmax().item()),
        "in_run_harmful_index": int(scores.argmin().item()),
        "in_run_helpful_index": int(scores.argmax().item()),
        "signal_correlation": exact_signal_correlation,
        "in_run_signal_correlation": in_run_signal_correlation,
        "max_abs_signal_correlation": max(
            abs(exact_signal_correlation),
            abs(in_run_signal_correlation),
        ),
    }


def random_data_attribution_failure_smoke_test() -> dict:
    """Return the deterministic random-data negative control."""
    signal_exact = exact_data_shapley_values(*toy_data_shapley_problem())
    train_x, train_y, val_x, val_y = _random_data_shapley_problem()
    metrics = _signal_failure_metrics(
        train_x,
        train_y,
        val_x,
        val_y,
        signal_exact=signal_exact,
    )
    original_harmful_index = int(signal_exact.argmin().item())
    original_helpful_index = int(signal_exact.argmax().item())
    metrics["random_data_attribution_fails"] = (
        metrics["harmful_index"] != original_harmful_index
        and metrics["helpful_index"] != original_helpful_index
        and metrics["max_abs_signal_correlation"] <= 0.25
    )
    metrics["original_harmful_index"] = original_harmful_index
    metrics["original_helpful_index"] = original_helpful_index
    metrics["seed"] = DATA_SHAPLEY_RANDOM_CONTROL_SEED
    return metrics


def label_shuffled_attribution_failure_smoke_test() -> dict:
    """Return the deterministic label-shuffle negative control."""
    signal_exact = exact_data_shapley_values(*toy_data_shapley_problem())
    train_x, train_y, val_x, val_y = _label_shuffled_data_shapley_problem()
    metrics = _signal_failure_metrics(
        train_x,
        train_y,
        val_x,
        val_y,
        signal_exact=signal_exact,
    )
    original_harmful_index = int(signal_exact.argmin().item())
    metrics["label_shuffled_attribution_fails"] = (
        metrics["harmful_index"] != original_harmful_index
        and metrics["in_run_harmful_index"] != original_harmful_index
        and metrics["signal_correlation"] <= 0.0
        and metrics["in_run_signal_correlation"] <= 0.0
    )
    metrics["original_harmful_index"] = original_harmful_index
    metrics["label_permutation"] = list(DATA_SHAPLEY_LABEL_SHUFFLE_PERMUTATION)
    metrics["shuffled_train_y"] = train_y.tolist()
    return metrics


if MAIN:
    tests.test_exact_data_shapley_smoke_test(exact_data_shapley_smoke_test)
    tests.test_monte_carlo_data_shapley_smoke_test(monte_carlo_data_shapley_smoke_test)
    tests.test_in_run_data_shapley_smoke_test(in_run_data_shapley_smoke_test)
    tests.test_random_data_attribution_failure_smoke_test(
        random_data_attribution_failure_smoke_test
    )
    tests.test_label_shuffled_attribution_failure_smoke_test(
        label_shuffled_attribution_failure_smoke_test
    )


All tests in `test_exact_data_shapley_smoke_test` passed!
All tests in `test_monte_carlo_data_shapley_smoke_test` passed!
All tests in `test_in_run_data_shapley_smoke_test` passed!
All tests in `test_random_data_attribution_failure_smoke_test` passed!
All tests in `test_label_shuffled_attribution_failure_smoke_test` passed!


## Exercise 6 - notebook contract and runtime

Package the local evidence and measure runtime overhead for full update, exact enumeration, and in-run proxy paths.

<details><summary>Expected output</summary>

```text
All tests in `test_runtime_overhead_smoke_test` passed!
All tests in `test_notebook_contract` passed!
```

</details>

<details><summary>Common bugs</summary>

- Returning tensors instead of lists.
- Omitting the negative controls from the contract.
- Reporting runtime without repeated measurements.

</details>

<details><summary>Solution</summary>

Return a dictionary with exact, Monte Carlo, in-run, both controls, and runtime-overhead metrics.

</details>


In [8]:
def _ratio(numerator: float, denominator: float) -> float:
    return numerator / max(denominator, 1e-12)


def _measure_wall_seconds(fn, *, repeats: int) -> float:
    if repeats <= 0:
        raise ValueError("repeats must be positive.")
    fn()
    start = time.perf_counter()
    for _ in range(repeats):
        fn()
    return (time.perf_counter() - start) / repeats


def runtime_overhead_smoke_test(repeats: int = DATA_SHAPLEY_RUNTIME_REPEATS) -> dict:
    """Measure full-update, exact-enumeration, and in-run score overhead."""
    train_x, train_y, val_x, val_y = toy_data_shapley_problem()
    full = frozenset(range(int(train_x.shape[0])))
    full_update_seconds = _measure_wall_seconds(
        lambda: one_step_linear_utility(train_x, train_y, val_x, val_y, full),
        repeats=repeats,
    )
    exact_enumeration_seconds = _measure_wall_seconds(
        lambda: data_coalition_values(train_x, train_y, val_x, val_y),
        repeats=repeats,
    )
    in_run_scores_seconds = _measure_wall_seconds(
        lambda: in_run_first_order_data_scores(train_x, train_y, val_x, val_y),
        repeats=repeats,
    )
    return {
        "runtime_measurement_repeats": repeats,
        "runtime_full_update_seconds": full_update_seconds,
        "runtime_exact_enumeration_seconds": exact_enumeration_seconds,
        "runtime_in_run_scores_seconds": in_run_scores_seconds,
        "runtime_exact_vs_full_update_overhead_ratio": _ratio(
            exact_enumeration_seconds,
            full_update_seconds,
        ),
        "runtime_in_run_vs_full_update_overhead_ratio": _ratio(
            in_run_scores_seconds,
            full_update_seconds,
        ),
        "runtime_in_run_vs_exact_ratio": _ratio(
            in_run_scores_seconds,
            exact_enumeration_seconds,
        ),
        "runtime_overhead_reported": (
            full_update_seconds > 0.0
            and exact_enumeration_seconds > 0.0
            and in_run_scores_seconds > 0.0
        ),
    }


def run_smoke_test(cpu: bool = True) -> dict:
    """Package the local Data Shapley evidence."""
    _ = cpu
    return {
        "exact": exact_data_shapley_smoke_test(),
        "monte_carlo": monte_carlo_data_shapley_smoke_test(),
        "in_run": in_run_data_shapley_smoke_test(),
        "random_data_control": random_data_attribution_failure_smoke_test(),
        "label_shuffle_control": label_shuffled_attribution_failure_smoke_test(),
        "runtime_overhead": runtime_overhead_smoke_test(),
    }


if MAIN:
    tests.test_runtime_overhead_smoke_test(runtime_overhead_smoke_test)
    tests.test_notebook_contract(run_smoke_test)


All tests in `test_runtime_overhead_smoke_test` passed!
All tests in `test_notebook_contract` passed!


## Exercise 7 - CUDA report interpretation

The committed report reruns the finite problem on CUDA, including exact enumeration, an actual optimizer step, in-run autograd scores, controls, and runtime.

<details><summary>Expected output</summary>

```text
All tests in `test_committed_gpu_report_records_exact_proxy_and_controls` passed!
{
  "exact_values": [0.6412, 0.6412, 0.6412, -1.1736],
  "sampled_max_abs_error": 0.0641,
  "gradient_scores": [4.0, 4.0, 4.0, -4.0],
  "random_data_attribution_fails": true,
  "label_shuffled_attribution_fails": true
}
```

</details>

<details><summary>Help - what does this prove?</summary>

It proves the exact/proxy/control pattern on a generated one-step CUDA model organism. It does not prove production data valuation, long-horizon optimizer influence, or real-dataset TracIn behavior.

</details>


In [9]:
def load_committed_gpu_report() -> dict:
    """Load the committed CUDA report for interpretation inside the notebook."""
    report = json.loads((section_dir / "verification_report.json").read_text())
    return report["metrics"]["gpu_test"]


def run_gpu_test(max_vram_gb: float = 24.0) -> dict:
    """Run the full CUDA experiment from the section solution module."""
    from chapter16_shapley_attribution_baselines.exercises.part7_data_shapley_in_one_training_run import solutions

    return solutions.run_gpu_test(max_vram_gb=max_vram_gb)


def run_full_experiment(max_vram_gb: float = 24.0) -> dict:
    """Alias used by the repository verification harness."""
    return run_gpu_test(max_vram_gb=max_vram_gb)


if MAIN:
    gpu_report = load_committed_gpu_report()
    tests.test_committed_gpu_report_records_exact_proxy_and_controls()
    summary = {
        "exact_values": gpu_report["exact_values"],
        "sampled_max_abs_error": gpu_report["sampled_max_abs_error"],
        "gradient_scores": gpu_report["gradient_scores"],
        "random_data_attribution_fails": gpu_report["random_data_attribution_fails"],
        "label_shuffled_attribution_fails": gpu_report["label_shuffled_attribution_fails"],
        "peak_vram_gb": gpu_report["peak_vram_gb"],
    }
    print(json.dumps(summary, indent=2))


All tests in `test_committed_gpu_report_records_exact_proxy_and_controls` passed!
{
  "exact_values": [
    0.6412037037037037,
    0.6412037037037037,
    0.6412037037037037,
    -1.173611111111111
  ],
  "sampled_max_abs_error": 0.0641276041666694,
  "gradient_scores": [
    4.0,
    4.0,
    4.0,
    -4.0
  ],
  "random_data_attribution_fails": true,
  "label_shuffled_attribution_fails": true,
  "peak_vram_gb": 0.06251144409179688
}


## Signature Result

<img src="../../instructions/assets/data_shapley_signature_result.svg" width="780">

| Example | Exact Data Shapley | Sampled estimate | In-run score | Interpretation |
|---:|---:|---:|---:|---|
| 0 | `0.6412` | `0.6613` | `4.0` | helpful |
| 1 | `0.6412` | `0.6403` | `4.0` | helpful |
| 2 | `0.6412` | `0.6861` | `4.0` | helpful |
| 3 | `-1.1736` | `-1.2377` | `-4.0` | flipped label, harmful |

<details><summary>What this section shows</summary>

- Exact Data Shapley identifies the flipped-label example as harmful.
- Sampled permutation Data Shapley preserves the harmful-example ranking.
- The one-run gradient-dot proxy correlates with exact values on this toy task.
- Random-data and label-shuffled controls fail, as they should.

</details>

## Limitations

<details><summary>What this section does not show</summary>

- It does not prove production-scale data valuation.
- It does not prove one-run proxies are exact for arbitrary optimizers or long training runs.
- It does not claim TracIn or influence-function parity.
- It does not use private, unsafe, or real user data.

</details>

## Bonus / anomaly hunting

- Increase the number of flipped labels and watch exact values shift.
- Add duplicated helpful examples and inspect how Shapley splits credit.
- Change the learning rate until the first-order proxy breaks.
- Replace squared loss with logistic loss and rerun the exact finite game.
